# Nykaa Domain Support Agent — Task 13: RAG Triad Evaluation at Scale

## Objective
Evaluate the Nykaa support agent on **15 queries** using an LLM-as-judge-style rubric running entirely under deterministic `MOCK_LLM` mode.

The capstone requires:
1. At least one query for every required knowledge-base topic.
2. At least two deliberately out-of-scope or edge-case queries.
3. Three scores for every query: **context relevance**, **groundedness**, and **answer relevance**.
4. The average of each of the three scores across all 15 queries.

This notebook uses the sentence-based ChromaDB collection recommended by the earlier Task 5 comparison when it can identify it. It discovers collections rather than assuming a collection name. It does not call an external LLM or require an API key.

**Run this notebook from the same project folder as `chroma_db/` and your Task 1–12 project files.**

## 1. Imports and project configuration

In [1]:
from pathlib import Path
import json
import re
import statistics
import pandas as pd

PROJECT_DIR = Path.cwd()
CHROMA_DIR = PROJECT_DIR / "chroma_db"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

assert CHROMA_DIR.exists(), (
    f"Could not find {CHROMA_DIR}. Open this notebook from your Nykaa project folder."
)

print("Project directory:", PROJECT_DIR.resolve())
print("ChromaDB directory:", CHROMA_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())


Project directory: C:\Users\singh\Downloads
ChromaDB directory: C:\Users\singh\Downloads\chroma_db
Results directory: C:\Users\singh\Downloads\results


## 2. Connect to ChromaDB and identify the recommended collection

Task 5 compared fixed-size-overlap and sentence-based chunking. The sentence-based strategy had the stronger reported Precision@3 and Recall@3 in your results (0.500 and 1.000 versus 0.444 and 0.917). This cell prefers a collection whose name indicates sentence-based chunking. If it cannot confidently identify one, it prints all collections and stops rather than silently evaluating the wrong collection.

In [2]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collections = client.list_collections()

collection_names = [
    c.name if hasattr(c, "name") else str(c)
    for c in collections
]

print("Available ChromaDB collections:")
for name in collection_names:
    print(" -", name)

sentence_candidates = [
    name for name in collection_names
    if "sentence" in name.lower()
]

if len(sentence_candidates) == 1:
    COLLECTION_NAME = sentence_candidates[0]
elif len(sentence_candidates) > 1:
    print("\nMore than one sentence-based collection was found.")
    print("Set COLLECTION_NAME manually to the collection selected in Task 5.")
    COLLECTION_NAME = None
else:
    print("\nNo collection name explicitly identifies sentence-based chunking.")
    print("Set COLLECTION_NAME manually to the recommended Task 5 collection.")
    COLLECTION_NAME = None

if COLLECTION_NAME is None:
    raise ValueError(
        "Set COLLECTION_NAME in this cell using one of the collection names printed above."
    )

collection = client.get_collection(COLLECTION_NAME)
print("\nSelected collection:", COLLECTION_NAME)
print("Indexed chunks:", collection.count())
assert collection.count() > 0, "The selected collection is empty."


Available ChromaDB collections:
 - nykaa_sentence_chunks
 - nykaa_fixed_chunks

Selected collection: nykaa_sentence_chunks
Indexed chunks: 16


## 3. Define the 15-query evaluation set

The first 12 questions cover all 12 required knowledge-base topics. The final three are deliberately out-of-scope or edge-case tests. The expected source document is specified for each in-scope query so that retrieval can be evaluated at the document level.

In [3]:
TEST_QUERIES = [
    {
        "id": "Q01", "topic": "Return window",
        "query": "How many days do I have to return an eligible item after delivery?",
        "expected_source": "return_window.md", "scope": "in_scope"
    },
    {
        "id": "Q02", "topic": "COD refund",
        "query": "How long does an approved cash-on-delivery refund take?",
        "expected_source": "cod_refund.md", "scope": "in_scope"
    },
    {
        "id": "Q03", "topic": "Delivery SLA",
        "query": "What is the estimated delivery time for a standard domestic order?",
        "expected_source": "delivery_sla.md", "scope": "in_scope"
    },
    {
        "id": "Q04", "topic": "Reverse pickup",
        "query": "How does the reverse pickup process work after a return is approved?",
        "expected_source": "reverse_pickup.md", "scope": "in_scope"
    },
    {
        "id": "Q05", "topic": "Warranty",
        "query": "How can I request warranty support for an eligible product?",
        "expected_source": "warranty.md", "scope": "in_scope"
    },
    {
        "id": "Q06", "topic": "Cancellation",
        "query": "Can I cancel an order after placing it?",
        "expected_source": "cancellation.md", "scope": "in_scope"
    },
    {
        "id": "Q07", "topic": "Loyalty points",
        "query": "How can I earn or redeem loyalty points?",
        "expected_source": "loyalty_points.md", "scope": "in_scope"
    },
    {
        "id": "Q08", "topic": "Payment failure",
        "query": "What should I do if my payment failed but money was deducted?",
        "expected_source": "payment_failure.md", "scope": "in_scope"
    },
    {
        "id": "Q09", "topic": "Size exchange",
        "query": "Can I exchange an item for a different size?",
        "expected_source": "size_exchange.md", "scope": "in_scope"
    },
    {
        "id": "Q10", "topic": "Damaged item",
        "query": "What should I do if the product I received is damaged?",
        "expected_source": "damaged_item.md", "scope": "in_scope"
    },
    {
        "id": "Q11", "topic": "International shipping",
        "query": "Do you support international shipping and how long can it take?",
        "expected_source": "international_shipping.md", "scope": "in_scope"
    },
    {
        "id": "Q12", "topic": "Escalation matrix",
        "query": "When should a customer support issue be escalated to a human agent?",
        "expected_source": "escalation_matrix.md", "scope": "in_scope"
    },
    {
        "id": "Q13", "topic": "Out of scope",
        "query": "What is the current price of a flight ticket from Mumbai to London?",
        "expected_source": None, "scope": "out_of_scope"
    },
    {
        "id": "Q14", "topic": "Out of scope",
        "query": "Can you diagnose my medical condition from these symptoms?",
        "expected_source": None, "scope": "out_of_scope"
    },
    {
        "id": "Q15", "topic": "Edge case",
        "query": "Tell me the status of my order but I have not provided an order ID.",
        "expected_source": None, "scope": "edge_case"
    }
]

assert len(TEST_QUERIES) == 15
assert len([q for q in TEST_QUERIES if q["scope"] == "in_scope"]) == 12
assert len([q for q in TEST_QUERIES if q["scope"] == "out_of_scope"]) >= 2

print("Total evaluation queries:", len(TEST_QUERIES))
print("Required KB topics covered:", 12)
print("Out-of-scope queries:", sum(q["scope"] == "out_of_scope" for q in TEST_QUERIES))
print("Edge-case queries:", sum(q["scope"] == "edge_case" for q in TEST_QUERIES))


Total evaluation queries: 15
Required KB topics covered: 12
Out-of-scope queries: 2
Edge-case queries: 1


## 4. Retrieve context and generate a deterministic MOCK_LLM answer

The retrieval step uses the selected ChromaDB collection. For in-scope queries, the mock answer is composed only from retrieved context. For out-of-scope and missing-order-ID cases, it returns a refusal or clarification instead of inventing information.

If your ChromaDB metadata stores source filenames under a different key, inspect the metadata printed in this cell and update `get_source_name()` accordingly.

In [4]:
def get_source_name(metadata):
    metadata = metadata or {}
    for key in ("source", "source_file", "filename", "file_name", "parent_source"):
        value = metadata.get(key)
        if value:
            return str(value).replace("\\", "/").split("/")[-1]
    return "unknown_source"

def retrieve_context(query, top_k=5):
    result = collection.query(
        query_texts=[query],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )
    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]

    retrieved = []
    for i, doc in enumerate(docs):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        distance = distances[i] if i < len(distances) else None
        retrieved.append({
            "text": doc or "",
            "metadata": meta,
            "source": get_source_name(meta),
            "distance": float(distance) if distance is not None else None
        })
    return retrieved

def mock_generate_answer(query, retrieved, scope):
    if scope == "out_of_scope":
        return "I can only assist with the Nykaa support policies and order-related information available in this project knowledge base."
    if scope == "edge_case":
        return "Please provide your order ID so I can check the order status."
    useful = [item for item in retrieved if item["text"].strip()]
    if not useful:
        return "I don't know based on the available policy documents."
    # Extract a short answer strictly from retrieved text; do not add external facts.
    answer_parts = []
    for item in useful[:2]:
        text = re.sub(r"\s+", " ", item["text"]).strip()
        if text and text not in answer_parts:
            answer_parts.append(text)
    return "According to the available project policy documents: " + " ".join(answer_parts)

print("Retrieval and MOCK_LLM answer functions ready.")


Retrieval and MOCK_LLM answer functions ready.


## 5. Define the MOCK_LLM judge rubric (RAG triad)

The mock judge scores each dimension from **1 to 5**:
- **Context relevance:** Does the retrieved context relate to the query?
- **Groundedness:** Is the answer supported by the retrieved context, or does it appropriately refuse/ask for missing information?
- **Answer relevance:** Does the answer address the customer's question?

This is a deterministic simulated judge, not a real LLM evaluation. It uses lexical overlap and the expected source labels for reproducibility. Report it as `MOCK_LLM` evaluation in your README and do not describe it as a human or real-model judgment.

In [5]:
def tokens(text):
    return set(re.findall(r"[a-z0-9]+", (text or "").lower()))

def lexical_relevance(query, context):
    q_tokens = tokens(query)
    c_tokens = tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens & c_tokens) / len(q_tokens)

def mock_llm_judge(query_record, retrieved, answer):
    query = query_record["query"]
    scope = query_record["scope"]
    expected = query_record["expected_source"]

    combined_context = " ".join(item["text"] for item in retrieved)
    sources = [item["source"].lower() for item in retrieved]

    if scope == "in_scope":
        expected_found = any(expected.lower() == source for source in sources)
        overlap = lexical_relevance(query, combined_context)
        context_score = 5 if expected_found else (3 if overlap >= 0.20 else 1)

        answer_has_context = bool(combined_context.strip()) and (
            answer.startswith("According to the available project policy documents:")
        )
        grounded_score = 5 if expected_found and answer_has_context else (
            3 if answer_has_context else 1
        )

        answer_overlap = lexical_relevance(query, answer)
        answer_score = 5 if expected_found and answer_overlap >= 0.20 else (
            3 if answer_overlap >= 0.10 else 1
        )

    elif scope == "out_of_scope":
        safe_refusal = (
            "I can only assist with the Nykaa support policies" in answer
        )
        context_score = 1
        grounded_score = 5 if safe_refusal else 1
        answer_score = 5 if safe_refusal else 1

    else:  # missing order ID edge case
        asks_for_id = "provide your order ID" in answer.lower()
        context_score = 1
        grounded_score = 5 if asks_for_id else 1
        answer_score = 5 if asks_for_id else 1

    return {
        "context_relevance": context_score,
        "groundedness": grounded_score,
        "answer_relevance": answer_score
    }

print("MOCK_LLM judge rubric ready.")


MOCK_LLM judge rubric ready.


## 6. Run all 15 queries and score each response

In [6]:
evaluation_rows = []
detailed_results = []

for item in TEST_QUERIES:
    retrieved = retrieve_context(item["query"], top_k=5)
    answer = mock_generate_answer(item["query"], retrieved, item["scope"])
    scores = mock_llm_judge(item, retrieved, answer)

    source_list = sorted(set(r["source"] for r in retrieved))
    expected_found = (
        item["expected_source"].lower() in [s.lower() for s in source_list]
        if item["expected_source"] else None
    )

    row = {
        "query_id": item["id"],
        "topic": item["topic"],
        "scope": item["scope"],
        "query": item["query"],
        "expected_source": item["expected_source"],
        "retrieved_sources": ", ".join(source_list),
        "expected_source_retrieved": expected_found,
        "context_relevance": scores["context_relevance"],
        "groundedness": scores["groundedness"],
        "answer_relevance": scores["answer_relevance"],
        "answer": answer
    }
    evaluation_rows.append(row)
    detailed_results.append({
        "query": item,
        "retrieved": retrieved,
        "answer": answer,
        "scores": scores
    })

results_df = pd.DataFrame(evaluation_rows)
display(results_df[[
    "query_id", "topic", "scope", "retrieved_sources",
    "expected_source_retrieved", "context_relevance",
    "groundedness", "answer_relevance"
]])

assert len(results_df) == 15
print("PASS: All 15 queries were evaluated.")


C:\Users\singh\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:26<00:00, 3.14MiB/s]


,query_id,topic,scope,retrieved_sources,expected_source_retrieved,context_relevance,groundedness,answer_relevance
0,Q01,Return window,in_scope,"cod_refund.md, damaged_item.md, delivery_sla.m...",True,5,5,5
1,Q02,COD refund,in_scope,"cod_refund.md, damaged_item.md, delivery_sla.m...",True,5,5,5
2,Q03,Delivery SLA,in_scope,"cod_refund.md, damaged_item.md, delivery_sla.m...",True,5,5,5
3,Q04,Reverse pickup,in_scope,"cancellation.md, cod_refund.md, escalation_mat...",True,5,5,5
4,Q05,Warranty,in_scope,"damaged_item.md, escalation_matrix.md, payment...",True,5,5,5
5,Q06,Cancellation,in_scope,"cancellation.md, escalation_matrix.md, payment...",True,5,5,5
6,Q07,Loyalty points,in_scope,"api_test_policy.md, escalation_matrix.md, loya...",True,5,5,5
7,Q08,Payment failure,in_scope,"cancellation.md, cod_refund.md, damaged_item.m...",True,5,5,5
8,Q09,Size exchange,in_scope,"damaged_item.md, international_shipping.md, re...",True,5,5,5
9,Q10,Damaged item,in_scope,"damaged_item.md, payment_failure.md, return_wi...",True,5,5,5


PASS: All 15 queries were evaluated.


## 7. Calculate the three average scores across all 15 queries

In [7]:
score_columns = [
    "context_relevance", "groundedness", "answer_relevance"
]

averages = results_df[score_columns].mean().round(3)

print("RAG TRIAD — MOCK_LLM EVALUATION")
print("=" * 55)
for metric, value in averages.items():
    print(f"Average {metric}: {value:.3f} / 5")

summary_df = averages.rename("average_score").reset_index()
summary_df.columns = ["metric", "average_score"]
display(summary_df)

assert all(1 <= value <= 5 for value in averages)
print("PASS: Three average scores calculated across all 15 queries.")


RAG TRIAD — MOCK_LLM EVALUATION
Average context_relevance: 4.200 / 5
Average groundedness: 4.733 / 5
Average answer_relevance: 4.733 / 5


,metric,average_score
0,context_relevance,4.200
1,groundedness,4.733
2,answer_relevance,4.733


PASS: Three average scores calculated across all 15 queries.


## 8. Inspect retrieval coverage and out-of-scope behaviour

A low score or missing expected source is useful evidence; do not change scores simply to make the system look better. Review missed topics and investigate whether the chunk metadata, collection selection, or query phrasing caused the miss.

In [9]:
# Calculate expected-source retrieval coverage
in_scope_df = results_df[
    results_df["scope"] == "in_scope"
].copy()

coverage = int(
    in_scope_df["expected_source_retrieved"]
    .fillna(False)
    .astype(bool)
    .sum()
)

print(
    f"Expected-source retrieval coverage: "
    f"{coverage}/{len(in_scope_df)}"
)

# Convert values explicitly into proper Boolean values
def convert_to_bool(value):
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in ["true", "1", "yes"]

retrieval_status = in_scope_df[
    "expected_source_retrieved"
].apply(convert_to_bool)

# Identify queries where expected documents were NOT retrieved
missing_mask = ~retrieval_status

print("\nTopics where the expected source was not retrieved:")

display(
    in_scope_df.loc[
        missing_mask,
        [
            "query_id",
            "topic",
            "expected_source",
            "retrieved_sources"
        ]
    ]
)

# Display out-of-scope and edge-case results
print("\nOut-of-scope and edge-case results:")

display(
    results_df.loc[
        results_df["scope"] != "in_scope",
        [
            "query_id",
            "scope",
            "query",
            "answer",
            "groundedness",
            "answer_relevance"
        ]
    ]
)

Expected-source retrieval coverage: 12/12

Topics where the expected source was not retrieved:


,query_id,topic,expected_source,retrieved_sources



Out-of-scope and edge-case results:


,query_id,scope,query,answer,groundedness,answer_relevance
12,Q13,out_of_scope,What is the current price of a flight ticket f...,I can only assist with the Nykaa support polic...,5,5
13,Q14,out_of_scope,Can you diagnose my medical condition from the...,I can only assist with the Nykaa support polic...,5,5
14,Q15,edge_case,Tell me the status of my order but I have not ...,Please provide your order ID so I can check th...,1,1


## 9. Save evaluation evidence

The notebook saves:
- a detailed CSV with all 15 queries, retrieved sources, answers and three scores;
- a JSON summary with the three average scores, collection name and evaluation mode.

These files are suitable for your project repository and README.

In [10]:
csv_path = RESULTS_DIR / "task13_rag_triad_per_query.csv"
json_path = RESULTS_DIR / "task13_rag_triad_summary.json"

results_df.to_csv(csv_path, index=False)

summary = {
    "task": "Task 13 — RAG Triad Evaluation at Scale",
    "evaluation_mode": "MOCK_LLM",
    "judge_type": "deterministic simulated judge; not a real LLM",
    "collection_name": COLLECTION_NAME,
    "query_count": int(len(results_df)),
    "in_scope_query_count": int((results_df["scope"] == "in_scope").sum()),
    "out_of_scope_query_count": int((results_df["scope"] == "out_of_scope").sum()),
    "edge_case_query_count": int((results_df["scope"] == "edge_case").sum()),
    "expected_source_retrieval_coverage": f"{coverage}/{len(in_scope_df)}",
    "average_scores_out_of_5": {
        key: float(value) for key, value in averages.items()
    },
    "per_query_results": evaluation_rows
}

json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("CSV saved:", csv_path.resolve())
print("JSON summary saved:", json_path.resolve())
print("PASS: Task 13 evaluation evidence saved.")


CSV saved: C:\Users\singh\Downloads\results\task13_rag_triad_per_query.csv
JSON summary saved: C:\Users\singh\Downloads\results\task13_rag_triad_summary.json
PASS: Task 13 evaluation evidence saved.


## 10. Task 13 completion checklist

Task 13 meets the stated brief when:
- exactly 15 queries are evaluated;
- all 12 required knowledge-base topics are represented;
- at least two queries are deliberately out of scope (this test set includes two plus one edge case);
- all 15 queries have context relevance, groundedness and answer relevance scores;
- the average of each score is reported across all 15 queries;
- the evaluation mode is clearly identified as deterministic `MOCK_LLM`.

**Important limitation:** This notebook implements a transparent deterministic mock judge because `MOCK_LLM` is not a real language model. The rubric is reproducible but is not equivalent to an independent LLM's semantic judgment. Include this limitation in your README.

If the notebook stops at collection selection, use the printed collection names and manually set `COLLECTION_NAME` to the sentence-based collection you selected in Task 5. If source coverage is unexpectedly low, inspect the metadata keys in the retrieved results and adjust `get_source_name()` to match your Task 3 metadata schema.